# Applio: экспериментальные вокодеры

Этот ноутбук разворачивает ветку exp/vocoders из форка egor125552/Applio в актуальном Google Colab.

Порядок запуска:

1. Установка.
2. Подключение Google Диска и, при необходимости, восстановление логов.
3. Проверка среды.
4. Запуск интерфейса.
5. После остановки интерфейса — сохранение логов на Диск.

Перед запуском выберите среду выполнения с GPU. Ветка экспериментальная: ноутбук проверяет окружение и импорты, но обучение конкретного вокодера всё равно требует подходящей конфигурации.

[Открыть этот ноутбук в Google Colab](https://colab.research.google.com/github/egor125552/Applio/blob/exp/vocoders/assets/Applio_Vocoders.ipynb)

In [ ]:
# @title 1. Установить Applio
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/egor125552/Applio.git"  # @param {type:"string"}
BRANCH = "exp/vocoders"  # @param {type:"string"}
DOWNLOAD_MODELS = True  # @param {type:"boolean"}

IS_COLAB = Path("/content").is_dir()
WORKSPACE = Path("/content") if IS_COLAB else Path.cwd()
REPO_DIR = WORKSPACE / "Applio"


def run(command, *, cwd=None):
    command = [str(part) for part in command]
    print("+", " ".join(command))
    return subprocess.run(command, cwd=cwd, check=True)


def normalized_remote(url):
    return url.rstrip("/").removesuffix(".git")


if (REPO_DIR / ".git").is_dir():
    current_remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
        text=True,
    ).strip()
    if normalized_remote(current_remote) != normalized_remote(REPO_URL):
        raise RuntimeError(
            f"{REPO_DIR} уже содержит другой репозиторий: {current_remote}"
        )

    tracked_changes = subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO_DIR),
            "status",
            "--porcelain",
            "--untracked-files=no",
        ],
        text=True,
    ).strip()
    if tracked_changes:
        raise RuntimeError(
            "В рабочей копии есть изменения. Сохраните их или перезапустите среду, "
            "чтобы установка ничего не затёрла."
        )

    run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH, "--depth", "1"])
    run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"])
else:
    run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            REPO_DIR,
        ]
    )

os.chdir(REPO_DIR)

if IS_COLAB:
    run(["apt-get", "update", "-qq"])
    run(
        [
            "apt-get",
            "install",
            "-y",
            "-qq",
            "ffmpeg",
            "portaudio19-dev",
        ]
    )

try:
    run([sys.executable, "-m", "uv", "--version"])
except (subprocess.CalledProcessError, ModuleNotFoundError):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])

run(
    [
        sys.executable,
        "-m",
        "uv",
        "pip",
        "install",
        "--system",
        "-r",
        REPO_DIR / "requirements.txt",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu121",
        "--index-strategy",
        "unsafe-best-match",
    ]
)

if DOWNLOAD_MODELS:
    run(
        [
            sys.executable,
            "core.py",
            "prerequisites",
            "--pretraineds_v1_f0",
            "False",
            "--pretraineds_v1_nof0",
            "False",
            "--pretraineds_v2_f0",
            "True",
            "--pretraineds_v2_nof0",
            "False",
            "--models",
            "True",
            "--exe",
            "True",
        ],
        cwd=REPO_DIR,
    )

print(f"Applio установлен: {REPO_DIR}")
print(f"Ветка: {BRANCH}")

In [ ]:
# @title 2. Подключить Google Диск
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Это не Google Colab: подключение Диска пропущено.")
else:
    drive.mount("/content/drive")
    print("Google Диск подключён.")

In [ ]:
# @title 3. Восстановить модели и логи с Google Диска
from pathlib import Path
import shutil

BACKUP_DIR = Path("/content/drive/MyDrive/ApplioBackup")
LOGS_DIR = REPO_DIR / "logs"

if not Path("/content/drive").is_mount():
    print("Google Диск не подключён. Ячейка ничего не изменила.")
elif not BACKUP_DIR.exists():
    print(f"Резервная копия пока не создана: {BACKUP_DIR}")
else:
    LOGS_DIR.mkdir(parents=True, exist_ok=True)
    restored = 0
    for source in BACKUP_DIR.iterdir():
        destination = LOGS_DIR / source.name
        if source.is_dir():
            shutil.copytree(source, destination, dirs_exist_ok=True)
        else:
            shutil.copy2(source, destination)
        restored += 1
    print(f"Восстановлено объектов: {restored}")

In [ ]:
# @title 4. Проверить установку и импорты вокодеров
from pathlib import Path
import importlib
import json
import sys

import numpy as np
import torch

if not (1, 26) <= tuple(int(part) for part in np.__version__.split(".")[:2]) < (2, 0):
    raise RuntimeError(f"Неподдерживаемая версия NumPy: {np.__version__}")

modules = [
    "rvc.lib.algorithm.synthesizers",
    "rvc.lib.algorithm.discriminators.CoMBD",
    "rvc.lib.algorithm.discriminators.SBD",
    "rvc.lib.algorithm.discriminators.cd",
    "rvc.lib.algorithm.discriminators.mbd",
    "rvc.lib.algorithm.discriminators.mpd",
    "rvc.lib.algorithm.discriminators.mpd2",
    "rvc.lib.algorithm.discriminators.mpd3",
    "rvc.lib.algorithm.discriminators.mpd_san",
    "rvc.lib.algorithm.discriminators.mrd",
    "rvc.lib.algorithm.discriminators.mrd2",
    "rvc.lib.algorithm.discriminators.msd",
    "rvc.lib.algorithm.discriminators.mssbcqtd",
    "rvc.lib.algorithm.discriminators.univnet",
    "rvc.lib.algorithm.generators.bigvgan",
    "rvc.lib.algorithm.generators.ddsp",
    "rvc.lib.algorithm.generators.ddsp_v2",
    "rvc.lib.algorithm.generators.ddsp_v3",
    "rvc.lib.algorithm.generators.hifigan",
    "rvc.lib.algorithm.generators.hifigan_aa",
    "rvc.lib.algorithm.generators.hifigan_cam",
    "rvc.lib.algorithm.generators.hifigan_nsf",
    "rvc.lib.algorithm.generators.hifigan_pqmf",
    "rvc.lib.algorithm.generators.hifigan_snake",
    "rvc.lib.algorithm.generators.hiftnet",
    "rvc.lib.algorithm.generators.hiftnet2",
    "rvc.lib.algorithm.generators.ringformer",
    "rvc.lib.algorithm.generators.velocity",
    "rvc.lib.algorithm.generators.vocos",
    "rvc.lib.algorithm.generators.vocos_v2",
    "rvc.lib.algorithm.generators.wavehax",
    "rvc.lib.algorithm.generators.wavehax1d",
    "rvc.lib.algorithm.generators.wavenext",
]

for module_name in modules:
    importlib.import_module(module_name)

for config_path in sorted((REPO_DIR / "rvc" / "configs").glob("v[12]/*.json")):
    with config_path.open(encoding="utf-8") as config_file:
        config = json.load(config_file)
    hop_length = config["data"]["hop_length"]
    upsample_factor = 1
    for rate in config["model"]["upsample_rates"]:
        upsample_factor *= rate
    if hop_length != upsample_factor:
        raise RuntimeError(
            f"{config_path}: hop_length={hop_length}, "
            f"а произведение upsample_rates={upsample_factor}"
        )

if DOWNLOAD_MODELS:
    required_files = [
        REPO_DIR / "rvc/models/predictors/rmvpe.pt",
        REPO_DIR / "rvc/models/embedders/contentvec/pytorch_model.bin",
        REPO_DIR / "rvc/models/pretraineds/pretrained_v2/f0G48k.pth",
    ]
    missing = [str(path) for path in required_files if not path.is_file()]
    if missing:
        raise FileNotFoundError("Не загрузились обязательные файлы:\n" + "\n".join(missing))

if IS_COLAB and not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не найден. В Colab выберите среду выполнения с GPU и повторите установку."
    )

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Импортировано модулей генераторов и дискриминаторов: {len(modules)}")
print("Проверка завершена без ошибок.")

In [ ]:
# @title 5. Реальный smoke-test вокодеров
from pathlib import Path
import nbformat
from nbclient import NotebookClient

REPO_DIR = Path("/content/Applio")
SMOKE_REPORT = Path("/content/applio-vocoders-smoke.executed.ipynb")

smoke_cells = [
    nbformat.v4.new_code_cell(
        "import sys, torch\n"
        "print('Python:', sys.version)\n"
        "print('PyTorch:', torch.__version__)\n"
        "print('CUDA:', torch.cuda.is_available())\n"
    ),
    nbformat.v4.new_code_cell(
        "import compileall\n"
        "from pathlib import Path\n"
        "for target in ('app.py', 'core.py', 'rvc', 'tabs'):\n"
        "    ok = compileall.compile_dir(target, quiet=1) if Path(target).is_dir() else compileall.compile_file(target, quiet=1)\n"
        "    if not ok:\n"
        "        raise RuntimeError(f'compileall failed: {target}')\n"
        "print('compileall: OK')\n"
    ),
    nbformat.v4.new_code_cell(
        "import importlib\n"
        "modules = [\n"
        "    'rvc.lib.algorithm.synthesizers',\n"
        "    'rvc.lib.algorithm.discriminators.CoMBD',\n"
        "    'rvc.lib.algorithm.discriminators.SBD',\n"
        "    'rvc.lib.algorithm.discriminators.mbd',\n"
        "    'rvc.lib.algorithm.discriminators.mpd',\n"
        "    'rvc.lib.algorithm.discriminators.mrd',\n"
        "    'rvc.lib.algorithm.discriminators.msd',\n"
        "    'rvc.lib.algorithm.discriminators.univnet',\n"
        "    'rvc.lib.algorithm.generators.bigvgan',\n"
        "    'rvc.lib.algorithm.generators.ddsp',\n"
        "    'rvc.lib.algorithm.generators.ddsp_v2',\n"
        "    'rvc.lib.algorithm.generators.ddsp_v3',\n"
        "    'rvc.lib.algorithm.generators.hifigan',\n"
        "    'rvc.lib.algorithm.generators.hifigan_aa',\n"
        "    'rvc.lib.algorithm.generators.hifigan_cam',\n"
        "    'rvc.lib.algorithm.generators.hifigan_nsf',\n"
        "    'rvc.lib.algorithm.generators.hifigan_pqmf',\n"
        "    'rvc.lib.algorithm.generators.hifigan_snake',\n"
        "    'rvc.lib.algorithm.generators.hiftnet',\n"
        "    'rvc.lib.algorithm.generators.hiftnet2',\n"
        "    'rvc.lib.algorithm.generators.ringformer',\n"
        "    'rvc.lib.algorithm.generators.velocity',\n"
        "    'rvc.lib.algorithm.generators.vocos',\n"
        "    'rvc.lib.algorithm.generators.vocos_v2',\n"
        "    'rvc.lib.algorithm.generators.wavehax',\n"
        "    'rvc.lib.algorithm.generators.wavehax1d',\n"
        "    'rvc.lib.algorithm.generators.wavenext',\n"
        "]\n"
        "for name in modules:\n"
        "    importlib.import_module(name)\n"
        "print(f'imports: {len(modules)} OK')\n"
    ),
    nbformat.v4.new_code_cell(
        "import torch\n"
        "from rvc.lib.algorithm.generators.hifigan import Generator\n"
        "from rvc.lib.algorithm.generators.hifigan_nsf import GeneratorNSF\n"
        "common = dict(\n"
        "    initial_channel=4,\n"
        "    resblock_kernel_sizes=[3],\n"
        "    resblock_dilation_sizes=[[1, 3, 5]],\n"
        "    upsample_rates=[2, 2],\n"
        "    upsample_initial_channel=16,\n"
        "    upsample_kernel_sizes=[4, 4],\n"
        "    gin_channels=0,\n"
        ")\n"
        "features = torch.randn(1, 4, 8)\n"
        "for generator, arguments in (\n"
        "    (Generator(**common), (features,)),\n"
        "    (GeneratorNSF(**common, sr=400), (features, torch.full((1, 8), 100.0))),\n"
        "):\n"
        "    audio = generator(*arguments)\n"
        "    assert audio.shape == (1, 1, 32), audio.shape\n"
        "    assert torch.isfinite(audio).all()\n"
        "print('HiFi-GAN forward passes: OK')\n"
    ),
]

smoke_notebook = nbformat.v4.new_notebook(
    cells=smoke_cells,
    metadata={
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3",
        }
    },
)

client = NotebookClient(
    smoke_notebook,
    timeout=900,
    kernel_name="python3",
    resources={"metadata": {"path": str(REPO_DIR)}},
    allow_errors=False,
)
client.execute()
nbformat.write(smoke_notebook, SMOKE_REPORT)

print("Smoke-test выполнен. Отчёт:", SMOKE_REPORT)

## Запуск

Метод gradio проще и используется по умолчанию. Localtunnel оставлен как запасной вариант. Ячейка работает, пока запущен интерфейс; чтобы перейти к сохранению, остановите её кнопкой остановки.

In [ ]:
# @title 5. Запустить Applio
import os
import subprocess
import sys
import urllib.request

METHOD = "gradio"  # @param ["gradio", "localtunnel"]
PORT = 6969

environment = os.environ.copy()
environment["GRADIO_SERVER_NAME"] = "0.0.0.0"
environment["GRADIO_SERVER_PORT"] = str(PORT)

command = [sys.executable, "app.py", "--port", str(PORT)]

if METHOD == "gradio":
    command.append("--share")
elif METHOD == "localtunnel":
    run(["npm", "install", "-g", "-q", "localtunnel"])
    try:
        password = urllib.request.urlopen(
            "https://ipv4.icanhazip.com",
            timeout=10,
        ).read().decode().strip()
        print(f"Пароль Localtunnel: {password}")
    except Exception as error:
        print(f"Не удалось получить пароль Localtunnel: {error}")
    subprocess.Popen(["lt", "--port", str(PORT)], env=environment)
else:
    raise ValueError(f"Неизвестный метод публикации: {METHOD}")

print("Запуск Applio. Остановите ячейку, когда закончите работу.")
subprocess.run(command, cwd=REPO_DIR, env=environment, check=True)

In [ ]:
# @title 6. Сохранить модели и логи на Google Диск
from pathlib import Path
import shutil

BACKUP_DIR = Path("/content/drive/MyDrive/ApplioBackup")
LOGS_DIR = REPO_DIR / "logs"

if not Path("/content/drive").is_mount():
    raise RuntimeError("Сначала подключите Google Диск.")
if not LOGS_DIR.exists():
    raise FileNotFoundError(f"Папка логов не найдена: {LOGS_DIR}")

BACKUP_DIR.mkdir(parents=True, exist_ok=True)
ignore_runtime_files = shutil.ignore_patterns(
    "mute",
    "reference",
    "zips",
    "mute_spin",
    "mute_spin-v2",
)
shutil.copytree(
    LOGS_DIR,
    BACKUP_DIR,
    dirs_exist_ok=True,
    ignore=ignore_runtime_files,
)
print(f"Логи сохранены: {BACKUP_DIR}")

In [ ]:
# @title 7. Скачать пользовательские предобученные G/D
from pathlib import Path
import urllib.request
from urllib.parse import urlparse

G_URL = ""  # @param {type:"string"}
D_URL = ""  # @param {type:"string"}

destination = REPO_DIR / "rvc/models/pretraineds/pretraineds_custom"
destination.mkdir(parents=True, exist_ok=True)

for label, url in (("G", G_URL), ("D", D_URL)):
    url = url.strip().replace("?download=true", "")
    if not url:
        print(f"{label}: ссылка не задана, пропуск.")
        continue
    output = destination / Path(urlparse(url).path).name
    urllib.request.urlretrieve(url, output)
    print(f"{label}: {output}")